# Reproducing — 12-Lead ECG Arrhythmia Classification (CPSC-2018)

This notebook reproduces the results of
**[ecg-arrhythmia-classification](https://github.com/Kevindic0214/ecg-arrhythmia-classification)**
end-to-end on a free Colab **GPU**:

1. Download the **real CPSC-2018** data from PhysioNet (the Challenge-2020 repackaging).
2. Preprocess it with the repo's **own** pipeline (`src/preprocessing.py`).
3. Train the repo's **CNN + Bi-GRU + Attention** model (`src/model.py`, 934,993 params)
   with the full protocol (Adam 1e-4, dropout 0.3, LR halved every 10 epochs,
   early stopping, cost-sensitive loss).
4. Evaluate: AUC / macro-F1 / accuracy + a normalized confusion matrix.

> **First:** `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Run all`.

### About the data download (please read)
PhysioNet **rate-limits bulk downloads** — hammering it returns HTTP 500 error pages.
So this notebook downloads **gently** (few parallel connections, retries, content checks)
and is **resumable**: if a run gets throttled and stops with too few records, just
**re-run the download cell** — it keeps the good files and fetches the rest.

- **Default:** a **representative sample** across all rhythm classes (`FULL_DATASET = False`,
  ~1,400 records) — fast and much less likely to be throttled. Good enough for a real,
  all-9-classes result.
- **Full report scale:** set `FULL_DATASET = True` (all 6,877 records) — closest to the
  paper, but expect throttling; re-run the cell a few times until it completes.
- **If PhysioNet keeps failing**, wait ~15–30 min (the block clears) or use the Kaggle
  mirror `bjoernjostein/china-12lead-ecg-challenge-database` (needs a `kaggle.json` token).

> **Labels.** PhysioNet stores CPSC-2018 with SNOMED-CT `Dx` codes in each `.hea`; we map
> them back to the original 9 CPSC classes.


In [ ]:
# 0. Check the GPU is on
!nvidia-smi -L || echo "No GPU — set Runtime > Change runtime type > GPU (T4)"


In [ ]:
# 1. Get the repo + dependencies (safe to re-run)
import os
if not os.path.isdir("ecg-arrhythmia-classification"):
    !git clone -q https://github.com/Kevindic0214/ecg-arrhythmia-classification.git
!pip install -q PyWavelets            # torch / numpy / scipy / sklearn / matplotlib ship with Colab
import sys
sys.path.insert(0, "/content/ecg-arrhythmia-classification/src")
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())


In [ ]:
# 2. Download real CPSC-2018 records from PhysioNet — gentle, content-checked, RESUMABLE.
#    Re-run this cell if it stops early (PhysioNet throttles bulk downloads with HTTP 500).
import glob, time, urllib.request, concurrent.futures as cf

DATA = "/content/data"; os.makedirs(DATA, exist_ok=True)
!rm -rf /content/data/cpsc_2018           # drop any partial recursive-wget artifacts
BASE = "https://physionet.org/files/challenge-2020/1.0.2/training/cpsc_2018"

FULL_DATASET = False          # False = representative ~1.4k-record sample; True = all 6,877 (report scale)
SUBSET_STEP  = 5              # sample every Nth record across g1..g7 so all classes appear
ids = list(range(1, 6878)) if FULL_DATASET else list(range(1, 6878, SUBSET_STEP))

def rec_url(i, ext):
    return f"{BASE}/g{(i - 1)//1000 + 1}/A{i:04d}{ext}"

def valid(path, ext):
    try:
        if ext == ".hea":
            with open(path) as f:
                return f.read(1) == "A"        # real header starts with the record name; a 500 page starts with '<'
        return os.path.getsize(path) > 1000    # real .mat is ~180 KB; a 500 page is ~177 bytes
    except Exception:
        return False

def fetch(i):
    rec = f"A{i:04d}"
    for ext in (".hea", ".mat"):
        dst = os.path.join(DATA, rec + ext)
        if os.path.exists(dst) and valid(dst, ext):
            continue                            # resume: keep good files
        for attempt in range(4):
            try:
                urllib.request.urlretrieve(rec_url(i, ext), dst)   # raises on HTTP 500
                if valid(dst, ext):
                    break
            except Exception:
                time.sleep(1.5 * (attempt + 1))  # back off to ride out throttling
        else:
            return None
    return rec

print(f"Downloading {len(ids)} records (6 parallel, retries) — re-run if throttled ...")
ok = 0
with cf.ThreadPoolExecutor(max_workers=6) as ex:
    for j, r in enumerate(ex.map(fetch, ids), 1):
        ok += bool(r)
        if j % 300 == 0:
            print(f"  {j}/{len(ids)} attempted, {ok} complete")
print("newly-complete this run:", ok)

heas = sorted({os.path.basename(p): p for p in glob.glob(f"{DATA}/**/*.hea", recursive=True)
               if valid(p, ".hea")}.values())
print(len(heas), "valid record headers on disk")
assert len(heas) >= 100, ("Too few records — PhysioNet is throttling this IP (HTTP 500). "
                          "Wait ~15 min and RE-RUN this cell (it resumes), or use the Kaggle mirror.")


In [ ]:
# 3. Parse labels (SNOMED -> 9 CPSC classes) and preprocess with the repo's pipeline.
#    Skips any record with a missing/broken .mat or an unmapped label.
import numpy as np
from scipy.io import loadmat
from preprocessing import preprocess_recording
from dataset import CLASS_NAMES

# CPSC order: Normal, AF, I-AVB, LBBB, RBBB, PAC, PVC, STD, STE
SNOMED_TO_IDX = {
    "426783006": 0,
    "164889003": 1,
    "270492004": 2,
    "164909002": 3,
    "59118001": 4,  "713427006": 4,
    "284470004": 5, "63593006": 5,
    "427172004": 6, "164884008": 6,
    "429622005": 7,
    "164931005": 8,
}

def label_of(hea):
    with open(hea) as f:
        for line in f:
            if line.startswith("# Dx:"):
                for c in (x.strip() for x in line.split(":", 1)[1].split(",")):
                    if c in SNOMED_TO_IDX:
                        return SNOMED_TO_IDX[c]
    return None

X, y, used, skipped = [], [], 0, 0
for hea in heas:
    mat = hea[:-4] + ".mat"
    idx = label_of(hea)
    if idx is None or not (os.path.exists(mat) and os.path.getsize(mat) > 1000):
        skipped += 1; continue
    try:
        sig = loadmat(mat)["val"].astype(np.float64) / 1000.0
    except Exception:
        skipped += 1; continue
    for w in preprocess_recording(sig):            # (n_windows, 12, 5000)
        X.append(w.astype(np.float32)); y.append(idx)
    used += 1
    if used % 500 == 0:
        print(f"  preprocessed {used} records ...")

n_classes = len(set(y))
assert used >= 100 and n_classes >= 5, (
    f"Only {used} usable records / {n_classes} classes — the download was incomplete "
    "(PhysioNet throttling). Re-run the download cell until it reports a few hundred records.")
X = np.stack(X); y = np.array(y, np.int64)
print(f"used {used} records, skipped {skipped}  ->  windows: {X.shape}")
for c in range(9):
    print(f"  {CLASS_NAMES[c]:7s}: {(y==c).sum()}")


In [ ]:
# 4. Train — full protocol on GPU (mirrors src/train.py)
import copy, torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from dataset import ECGWindowDataset, class_weights
from model import ECGArrhythmiaNet

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
tr = DataLoader(ECGWindowDataset(Xtr, ytr), batch_size=32, shuffle=True)
va = DataLoader(ECGWindowDataset(Xva, yva), batch_size=64)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ECGArrhythmiaNet().to(dev)
crit = nn.CrossEntropyLoss(weight=class_weights(ytr).to(dev))   # cost-sensitive
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

EPOCHS, PATIENCE = 100, 10
hist = {"tr_loss": [], "va_loss": [], "va_acc": []}
best, best_state, wait = float("inf"), None, 0

def run(loader, train):
    model.train(train); tot = cor = n = 0.0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(dev), yb.to(dev)
            logits = model(xb); loss = crit(logits, yb)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(yb); cor += (logits.argmax(1)==yb).sum().item(); n += len(yb)
    return tot/n, cor/n

for ep in range(1, EPOCHS+1):
    trl, _ = run(tr, True)
    val, vac = run(va, False)
    sched.step()
    hist["tr_loss"].append(trl); hist["va_loss"].append(val); hist["va_acc"].append(vac)
    print(f"[{ep:3d}] train_loss {trl:.4f} | val_loss {val:.4f} val_acc {vac:.4f}")
    if val < best:
        best, best_state, wait = val, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {ep} (best val_loss {best:.4f})"); break
model.load_state_dict(best_state)


In [ ]:
# 5. Evaluate — AUC / macro-F1 / accuracy + confusion matrix
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

model.eval(); probs, tgts = [], []
with torch.no_grad():
    for xb, yb in va:
        probs.append(torch.softmax(model(xb.to(dev)), 1).cpu().numpy()); tgts.append(yb.numpy())
probs = np.concatenate(probs); tgts = np.concatenate(tgts); preds = probs.argmax(1)

present = sorted(set(tgts))
acc = accuracy_score(tgts, preds)
f1  = f1_score(tgts, preds, average="macro", labels=present)
try:
    auc = roc_auc_score(np.eye(9)[tgts][:, present], probs[:, present], average="macro", multi_class="ovr")
except ValueError:
    auc = float("nan")
print(f"Accuracy {acc:.4f} | macro-F1 {f1:.4f} | macro-AUC {auc:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(hist["tr_loss"], label="train loss"); ax[0].plot(hist["va_loss"], label="val loss")
ax[0].plot(hist["va_acc"], label="val acc"); ax[0].legend(); ax[0].set_xlabel("epoch"); ax[0].set_title("Training")
cm = confusion_matrix(tgts, preds, labels=list(range(9)), normalize="true")
im = ax[1].imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax[1].set_xticks(range(9)); ax[1].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax[1].set_yticks(range(9)); ax[1].set_yticklabels(CLASS_NAMES)
ax[1].set_title("Normalized confusion matrix"); fig.colorbar(im, ax=ax[1])
for i in range(9):
    for j in range(9):
        ax[1].text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                   color="white" if cm[i,j] > 0.5 else "black", fontsize=7)
plt.tight_layout(); plt.show()
